# 02_bronze

Reads from all raw bronze tables, applies cleaning and standardisation
transformations, and writes clean bronze Delta tables ready for silver joining.

This is NOT feature engineering — only cleaning:
  - Fix types
  - Handle nulls
  - Standardise column names
  - Remove obvious bad data
  - Add data quality flags

Input tables:
  precursor.bronze.alpaca_raw
  precursor.bronze.fred_raw
  precursor.bronze.sec_raw
  precursor.bronze.universe

Output tables:
  precursor.bronze.alpaca_clean
  precursor.bronze.fred_clean
  precursor.bronze.sec_clean

In [0]:
%pip install "numpy<2.0" --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, LongType,
    TimestampType, BooleanType, IntegerType,
)
from pyspark.sql.window import Window
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.bronze")

START_DATE = "2020-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

logger.info("START_DATE=%s  END_DATE=%s", START_DATE, END_DATE)

INFO:precursor.bronze:START_DATE=2020-01-01  END_DATE=2026-05-01


## Cell 3 — Data quality check helper

In [0]:
def log_quality_report(
    df: DataFrame,
    table_name: str,
    key_columns: list[str],
) -> dict:
    """Log a formatted data quality report for a raw table.

    Args:
        df:          Spark DataFrame to inspect.
        table_name:  Human-readable name used in log output.
        key_columns: Columns to check for nulls and duplicates.

    Returns:
        Dict with keys: total_rows, nulls_per_column, duplicate_count.
    """
    total_rows = df.count()

    nulls_per_column = {}
    for col in key_columns:
        n = df.filter(F.col(col).isNull()).count()
        nulls_per_column[col] = n

    duplicate_count = total_rows - df.dropDuplicates(key_columns).count()

    logger.info("=== %s Quality Report ===", table_name)
    logger.info("  Total rows      : %d", total_rows)
    for col, n in nulls_per_column.items():
        logger.info("  Nulls in %-20s: %d", col, n)
    logger.info("  Duplicates on %s: %d", key_columns, duplicate_count)

    return {
        "total_rows":        total_rows,
        "nulls_per_column":  nulls_per_column,
        "duplicate_count":   duplicate_count,
    }

## Cell 4 — Clean Alpaca OHLCV

In [0]:
def clean_alpaca(raw_df: DataFrame) -> DataFrame:
    """Clean and validate the Alpaca OHLCV raw table.

    Applies type casting, date filtering, universe join, null handling,
    OHLC relationship validation, volume checks, and split-detection flags.
    Flags are added rather than rows dropped (except null close) so
    downstream layers can make their own filtering decisions.

    Args:
        raw_df: Spark DataFrame read from precursor.bronze.alpaca_raw.

    Returns:
        Cleaned Spark DataFrame ready for precursor.bronze.alpaca_clean.
    """
    raw_count = raw_df.count()
    logger.info("alpaca_raw input rows: %d", raw_count)

    # Step 1: Cast and standardise types.
    # Ticker whitespace and case inconsistencies cause silent join misses downstream.
    df = (
        raw_df
        .withColumn("ticker", F.upper(F.trim(F.col("ticker").cast(StringType()))))
        .withColumn("date",   F.col("date").cast(DateType()))
        .withColumn("open",   F.col("open").cast(DoubleType()))
        .withColumn("high",   F.col("high").cast(DoubleType()))
        .withColumn("low",    F.col("low").cast(DoubleType()))
        .withColumn("close",  F.col("close").cast(DoubleType()))
        .withColumn("vwap",   F.col("vwap").cast(DoubleType()))
        .withColumn("volume", F.col("volume").cast(LongType()))
    )

    # Step 2: Filter to our analysis date window.
    df = df.filter(
        (F.col("date") >= F.lit(START_DATE).cast(DateType())) &
        (F.col("date") <= F.lit(END_DATE).cast(DateType()))
    )

    # Step 3: Inner join with universe on (ticker, date).
    # This removes rows for days the stock was not in the index or was not a
    # trading day. Using universe as the source of truth for valid ticker-dates
    # prevents phantom rows from leaking into downstream features.
    universe = spark.table("precursor.bronze.universe").select("ticker", "date")
    df = df.join(universe, on=["ticker", "date"], how="inner")

    # Step 4: Handle null prices.
    # Close is the primary target source — rows without it are unusable.
    # For open/high/low/vwap, filling with close is conservative but correct:
    # it avoids inflating missing-data counts while keeping the row usable.
    null_close_count = df.filter(F.col("close").isNull()).count()
    logger.info("alpaca: dropping %d rows with null close.", null_close_count)
    df = df.filter(F.col("close").isNotNull())

    for price_col in ("open", "high", "low", "vwap"):
        filled = df.filter(F.col(price_col).isNull()).count()
        if filled > 0:
            logger.info("alpaca: filling %d null %s values with close.", filled, price_col)
        df = df.withColumn(price_col, F.coalesce(F.col(price_col), F.col("close")))

    # Step 5: Validate OHLC relationships.
    # Violations indicate bad data from Alpaca (e.g. adjustment errors).
    # We flag rather than drop so analysts can investigate.
    df = df.withColumn(
        "ohlc_valid",
        (F.col("high") >= F.col("close")) &
        (F.col("close") >= F.col("low")) &
        (F.col("high") >= F.col("open")) &
        (F.col("open") >= F.col("low")),
    )
    invalid_ohlc = df.filter(~F.col("ohlc_valid")).count()
    logger.info("alpaca: %d rows with invalid OHLC relationships (flagged).", invalid_ohlc)

    # Step 6: Flag zero-volume rows.
    # Zero volume on a trading day suggests a data gap or halt. Downstream
    # models should not treat these as normal days.
    df = df.withColumn("zero_volume", F.col("volume") == 0)
    zero_vol = df.filter(F.col("zero_volume")).count()
    logger.info("alpaca: %d rows with zero volume (flagged).", zero_vol)

    # Step 7: Flag extreme single-day price moves as potential split leakage.
    # A >40% move in one day almost certainly indicates an unadjusted split
    # or corporate action that slipped through the adjustment pipeline.
    # We flag rather than drop because some extreme moves are genuine (halts,
    # acquisitions) and require human review.
    price_window = Window.partitionBy("ticker").orderBy("date")
    df = df.withColumn(
        "extreme_move",
        F.abs(F.col("close") / F.lag("close", 1).over(price_window) - 1) > 0.40,
    )
    # First row per ticker has null lag — treat as not extreme
    df = df.withColumn(
        "extreme_move",
        F.when(F.col("extreme_move").isNull(), F.lit(False)).otherwise(F.col("extreme_move")),
    )
    extreme_count = df.filter(F.col("extreme_move")).count()
    logger.info("alpaca: %d rows flagged as extreme moves (>40%% daily change).", extreme_count)

    # Step 8: Deduplicate on (ticker, date), keeping latest ingested_at.
    dedup_window = Window.partitionBy("ticker", "date").orderBy(F.desc("ingested_at"))
    df = (
        df
        .withColumn("_rn", F.row_number().over(dedup_window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # Step 9: Add pipeline metadata.
    df = (
        df
        .withColumn("source",       F.lit("alpaca"))
        .withColumn("processed_at", F.current_timestamp())
    )

    keep = [
        "ticker", "date", "open", "high", "low", "close", "volume", "vwap",
        "adjusted", "ohlc_valid", "zero_volume", "extreme_move",
        "source", "processed_at",
    ]
    df = df.select(*keep)

    clean_count = df.count()
    retained_pct = clean_count / raw_count * 100 if raw_count > 0 else 0
    logger.info(
        "alpaca_clean: %d rows retained (%.1f%% of %d raw rows).",
        clean_count, retained_pct, raw_count,
    )
    return df

## Cell 5 — Clean FRED

In [0]:
# FRED valid value ranges — out-of-range values indicate bad data or unit
# mismatches (e.g. CPI reported as a percentage instead of index level).
_FRED_RANGES = {
    "DFF":      (0.0,   25.0),
    "UNRATE":   (0.0,   25.0),
    "CPIAUCSL": (50.0,  400.0),
    "T10Y2Y":   (-5.0,  5.0),
    "VIXCLS":   (0.0,   100.0),
    "M2SL":     (0.0,   100_000.0),
}


def clean_fred(raw_df: DataFrame) -> DataFrame:
    """Clean and validate the FRED macroeconomic indicators raw table.

    Applies type casting, date filtering, forward-fill of remaining nulls
    within each series, and range validation flags.

    Args:
        raw_df: Spark DataFrame read from precursor.bronze.fred_raw.

    Returns:
        Cleaned Spark DataFrame ready for precursor.bronze.fred_clean.
    """
    raw_count = raw_df.count()
    logger.info("fred_raw input rows: %d", raw_count)

    # Step 1: Cast types.
    df = (
        raw_df
        .withColumn("series_id", F.col("series_id").cast(StringType()))
        .withColumn("date",      F.col("date").cast(DateType()))
        .withColumn("value",     F.col("value").cast(DoubleType()))
    )

    # Step 2: Filter date range.
    df = df.filter(
        (F.col("date") >= F.lit(START_DATE).cast(DateType())) &
        (F.col("date") <= F.lit(END_DATE).cast(DateType()))
    )

    # Step 3: Forward-fill nulls within each series.
    # FRED occasionally has missing observations for daily series on holidays.
    # Forward-fill within the series is the correct economic interpretation —
    # the last known value persists until the next publication.
    fred_window = (
        Window.partitionBy("series_id")
        .orderBy("date")
        .rowsBetween(Window.unboundedPreceding, 0)
    )
    null_before = df.filter(F.col("value").isNull()).count()
    df = df.withColumn("value", F.last("value", ignorenulls=True).over(fred_window))
    null_after = df.filter(F.col("value").isNull()).count()
    logger.info(
        "fred: forward-filled %d null values (%d remaining unfillable).",
        null_before - null_after, null_after,
    )

    # Ensure is_forward_filled column exists (bootstrap sets it, verify here).
    if "is_forward_filled" not in df.columns:
        df = df.withColumn("is_forward_filled", F.lit(False))

    # Step 4: Validate value ranges.
    # Build a CASE WHEN expression that checks each series against its expected
    # range. Out-of-range values indicate unit errors or data corruption.
    range_expr = F.lit(True)
    for series_id, (lo, hi) in _FRED_RANGES.items():
        range_expr = F.when(
            F.col("series_id") == series_id,
            (F.col("value") >= lo) & (F.col("value") <= hi),
        ).otherwise(range_expr)

    df = df.withColumn("range_valid", range_expr)
    out_of_range = df.filter(~F.col("range_valid")).count()
    logger.info("fred: %d rows with out-of-range values (flagged).", out_of_range)

    # Step 5: Deduplicate on (series_id, date), keeping latest ingested_at.
    dedup_window = Window.partitionBy("series_id", "date").orderBy(F.desc("ingested_at"))
    df = (
        df
        .withColumn("_rn", F.row_number().over(dedup_window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # Step 6: Add pipeline metadata.
    df = (
        df
        .withColumn("source",       F.lit("fred"))
        .withColumn("processed_at", F.current_timestamp())
    )

    keep = [
        "series_id", "description", "date", "value", "frequency",
        "is_forward_filled", "range_valid", "source", "processed_at",
    ]
    df = df.select(*keep)

    clean_count = df.count()
    retained_pct = clean_count / raw_count * 100 if raw_count > 0 else 0
    logger.info(
        "fred_clean: %d rows retained (%.1f%% of %d raw rows).",
        clean_count, retained_pct, raw_count,
    )
    return df

## Cell 6 — Clean SEC

In [0]:
def clean_sec(raw_df: DataFrame) -> DataFrame:
    """Clean and validate the SEC Form 4 insider trading filings raw table.

    Applies type casting, date filtering, universe join, date-order validation,
    and late-filing flags.

    Args:
        raw_df: Spark DataFrame read from precursor.bronze.sec_raw.

    Returns:
        Cleaned Spark DataFrame ready for precursor.bronze.sec_clean.
    """
    raw_count = raw_df.count()
    logger.info("sec_raw input rows: %d", raw_count)

    # Step 1: Cast types.
    df = (
        raw_df
        .withColumn("transaction_date", F.col("transaction_date").cast(DateType()))
        .withColumn("filing_date",      F.col("filing_date").cast(DateType()))
        .withColumn("days_to_file",     F.col("days_to_file").cast(LongType()))
        .withColumn("is_late_filing",   F.col("is_late_filing").cast(BooleanType()))
    )

    # Step 2: Filter to analysis date window on transaction_date.
    # We use transaction_date (reportDate) not filing_date to avoid look-ahead
    # bias — the same reason it was chosen in the bootstrap.
    df = df.filter(
        (F.col("transaction_date") >= F.lit(START_DATE).cast(DateType())) &
        (F.col("transaction_date") <= F.lit(END_DATE).cast(DateType()))
    )

    # Step 3: Validate date order.
    # A filing cannot precede its own transaction — if it does, the data is
    # corrupted or the dates were swapped in the source XML.
    df = df.withColumn(
        "date_order_valid",
        F.col("transaction_date") <= F.col("filing_date"),
    )
    invalid_order = df.filter(~F.col("date_order_valid")).count()
    logger.info("sec: %d rows with transaction_date > filing_date (flagged).", invalid_order)

    # Step 4: Filter to valid tickers only.
    # Tickers not in our universe are out of scope — they would create
    # unmatched rows in all downstream silver joins.
    universe_tickers = (
        spark.table("precursor.bronze.universe")
        .select("ticker")
        .distinct()
    )
    before_join = df.count()
    df = df.join(universe_tickers, on="ticker", how="inner")
    dropped = before_join - df.count()
    logger.info("sec: dropped %d rows for tickers not in universe.", dropped)

    # Step 5: Flag very late filings.
    # SEC requires Form 4 within 2 business days. Filings >10 days late are
    # suspicious — they may represent amended filings, restatements, or data
    # errors. We flag rather than drop because late filings are still valid
    # insider trading signals.
    df = df.withColumn("very_late_filing", F.col("days_to_file") > 10)
    very_late = df.filter(F.col("very_late_filing")).count()
    logger.info("sec: %d rows flagged as very late (days_to_file > 10).", very_late)

    # Step 6: Verify no duplicate accession numbers.
    # accession_number is the SEC's unique filing identifier — duplicates
    # indicate double-ingestion and would inflate insider activity counts.
    total      = df.count()
    distinct   = df.dropDuplicates(["accession_number"]).count()
    dup_count  = total - distinct
    logger.info("sec: %d duplicate accession_numbers found.", dup_count)
    if dup_count > 0:
        dedup_window = Window.partitionBy("accession_number").orderBy(F.desc("transaction_date"))
        df = (
            df
            .withColumn("_rn", F.row_number().over(dedup_window))
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

    # Step 7: Add pipeline metadata.
    df = (
        df
        .withColumn("source",       F.lit("sec"))
        .withColumn("processed_at", F.current_timestamp())
    )

    keep = [
        "ticker", "cik", "accession_number", "transaction_date", "filing_date",
        "form_type", "days_to_file", "is_late_filing", "very_late_filing",
        "date_order_valid", "source", "processed_at",
    ]
    df = df.select(*keep)

    clean_count = df.count()
    retained_pct = clean_count / raw_count * 100 if raw_count > 0 else 0
    logger.info(
        "sec_clean: %d rows retained (%.1f%% of %d raw rows).",
        clean_count, retained_pct, raw_count,
    )
    return df

## Cell 7 — Main execution

In [0]:
_pipeline_start = datetime.now()
logger.info("=== precursor.02_bronze START at %s ===", _pipeline_start.isoformat())

_summary: list[tuple[str, int, int]] = []  # (table, raw_rows, clean_rows)

try:
    # --- Alpaca ---
    logger.info("--- Processing alpaca ---")
    _alpaca_raw = spark.table("precursor.bronze.alpaca_raw")
    _alpaca_raw_count = _alpaca_raw.count()
    log_quality_report(_alpaca_raw, "alpaca_raw", ["ticker", "date"])
    _alpaca_clean = clean_alpaca(_alpaca_raw)
    _alpaca_clean_count = _alpaca_clean.count()
    (
        _alpaca_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.bronze.alpaca_clean")
    )
    logger.info("precursor.bronze.alpaca_clean written.")
    _summary.append(("alpaca_clean", _alpaca_raw_count, _alpaca_clean_count))

    # --- FRED ---
    logger.info("--- Processing FRED ---")
    _fred_raw = spark.table("precursor.bronze.fred_raw")
    _fred_raw_count = _fred_raw.count()
    log_quality_report(_fred_raw, "fred_raw", ["series_id", "date"])
    _fred_clean = clean_fred(_fred_raw)
    _fred_clean_count = _fred_clean.count()
    (
        _fred_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.bronze.fred_clean")
    )
    logger.info("precursor.bronze.fred_clean written.")
    _summary.append(("fred_clean", _fred_raw_count, _fred_clean_count))

    # --- SEC ---
    logger.info("--- Processing SEC ---")
    _sec_raw = spark.table("precursor.bronze.sec_raw")
    _sec_raw_count = _sec_raw.count()
    log_quality_report(_sec_raw, "sec_raw", ["accession_number"])
    _sec_clean = clean_sec(_sec_raw)
    _sec_clean_count = _sec_clean.count()
    (
        _sec_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.bronze.sec_clean")
    )
    logger.info("precursor.bronze.sec_clean written.")
    _summary.append(("sec_clean", _sec_raw_count, _sec_clean_count))

    _elapsed = (datetime.now() - _pipeline_start).total_seconds() / 60
    logger.info("=== precursor.02_bronze END — %.2f min total ===", _elapsed)

    print("=" * 65)
    print("  PRECURSOR — BRONZE CLEANING SUMMARY")
    print("=" * 65)
    print(f"  {'Table':<20} {'Raw Rows':>12} {'Clean Rows':>12} {'Retained%':>10}")
    print("-" * 65)
    for table, raw, clean in _summary:
        pct = clean / raw * 100 if raw > 0 else 0
        print(f"  {table:<20} {raw:>12,} {clean:>12,} {pct:>9.1f}%")
    print("=" * 65)
    print(f"  Elapsed time: {_elapsed:.2f} min")
    print("=" * 65)

except Exception as exc:
    logger.error("Bronze pipeline failed: %s", exc, exc_info=True)

INFO:precursor.bronze:=== precursor.02_bronze START at 2026-05-01T18:58:38.328972 ===
INFO:precursor.bronze:--- Processing alpaca ---
INFO:precursor.bronze:=== alpaca_raw Quality Report ===
INFO:precursor.bronze:  Total rows      : 917749
INFO:precursor.bronze:  Nulls in ticker              : 0
INFO:precursor.bronze:  Nulls in date                : 0
INFO:precursor.bronze:  Duplicates on ['ticker', 'date']: 0
INFO:precursor.bronze:alpaca_raw input rows: 917749
INFO:precursor.bronze:alpaca: dropping 0 rows with null close.
INFO:precursor.bronze:alpaca: 0 rows with invalid OHLC relationships (flagged).
INFO:precursor.bronze:alpaca: 1336 rows with zero volume (flagged).
INFO:precursor.bronze:alpaca: 56 rows flagged as extreme moves (>40% daily change).
INFO:precursor.bronze:alpaca_clean: 917749 rows retained (100.0% of 917749 raw rows).
INFO:precursor.bronze:precursor.bronze.alpaca_clean written.
INFO:precursor.bronze:--- Processing FRED ---
INFO:precursor.bronze:=== fred_raw Quality Repo

  PRECURSOR — BRONZE CLEANING SUMMARY
  Table                    Raw Rows   Clean Rows  Retained%
-----------------------------------------------------------------
  alpaca_clean              917,749      917,749     100.0%
  fred_clean                 12,387       12,387     100.0%
  sec_clean                 225,346      225,346     100.0%
  Elapsed time: 0.65 min


## Cell 8 — Validation

In [0]:
logger.info("=== Running bronze validation ===")

_checks: list[tuple[str, bool, str]] = []

def _check(name: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
    logger.info(msg)
    _checks.append((name, passed, detail))


try:
    from datetime import timedelta

    # ── alpaca_clean ────────────────────────────────────────────────
    _a = spark.table("precursor.bronze.alpaca_clean")

    _row_count = _a.count()
    _check("alpaca_clean: row count > 800,000", _row_count > 800_000, f"{_row_count:,} rows")

    _null_close = _a.filter(F.col("close").isNull()).count()
    _check("alpaca_clean: no null close prices", _null_close == 0, f"{_null_close} nulls")

    _bad_dates = _a.filter(F.col("date") < "2020-01-01").count()
    _check("alpaca_clean: all dates >= 2020-01-01", _bad_dates == 0, f"{_bad_dates} pre-2020 rows")

    _distinct_tickers = _a.select("ticker").distinct().count()
    _check("alpaca_clean: distinct tickers >= 500", _distinct_tickers >= 500, f"{_distinct_tickers} tickers")

    _ohlc_valid_pct = _a.filter(F.col("ohlc_valid")).count() / _row_count * 100
    _check("alpaca_clean: ohlc_valid > 99%", _ohlc_valid_pct > 99, f"{_ohlc_valid_pct:.2f}%")

    _extreme_pct = _a.filter(F.col("extreme_move")).count() / _row_count * 100
    _check("alpaca_clean: extreme_move < 1%", _extreme_pct < 1, f"{_extreme_pct:.2f}%")

    # ── fred_clean ──────────────────────────────────────────────────
    _f = spark.table("precursor.bronze.fred_clean")

    _series_count = _f.select("series_id").distinct().count()
    _check("fred_clean: exactly 6 series_ids", _series_count == 6, f"{_series_count} found")

    _null_values = _f.filter(F.col("value").isNull()).count()
    _check("fred_clean: no null values", _null_values == 0, f"{_null_values} nulls")

    _out_of_range = _f.filter(~F.col("range_valid")).count()
    _check("fred_clean: all range_valid = TRUE", _out_of_range == 0, f"{_out_of_range} out-of-range rows")

    _fred_latest = _f.agg(F.max("date").alias("d")).collect()[0]["d"]
    _five_days_ago = (datetime.today() - timedelta(days=5)).date()
    _check(
        "fred_clean: latest date within 5 days",
        _fred_latest is not None and _fred_latest >= _five_days_ago,
        f"latest = {_fred_latest}",
    )

    # ── sec_clean ───────────────────────────────────────────────────
    _s = spark.table("precursor.bronze.sec_clean")

    _sec_rows = _s.count()
    _check("sec_clean: row count > 100,000", _sec_rows > 100_000, f"{_sec_rows:,} rows")

    _null_tx_dates = _s.filter(F.col("transaction_date").isNull()).count()
    _check("sec_clean: no null transaction_dates", _null_tx_dates == 0, f"{_null_tx_dates} nulls")

    _date_order_pct = _s.filter(F.col("date_order_valid")).count() / _sec_rows * 100
    _check("sec_clean: date_order_valid > 99%", _date_order_pct > 99, f"{_date_order_pct:.2f}%")

    _sec_tickers = _s.select("ticker").distinct().count()
    _check("sec_clean: distinct tickers >= 400", _sec_tickers >= 400, f"{_sec_tickers} tickers")

except Exception as exc:
    logger.error("Validation query failed: %s", exc, exc_info=True)
    _checks.append(("Validation queries executed without error", False, str(exc)))

_passed_n = sum(1 for _, p, _ in _checks if p)
_failed_n = len(_checks) - _passed_n

print("=" * 60)
print("  VALIDATION RESULTS")
print("=" * 60)
for name, passed, detail in _checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {_passed_n} passed  /  {_failed_n} failed")
if _failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.bronze:=== Running bronze validation ===
INFO:precursor.bronze:[PASS] alpaca_clean: row count > 800,000 — 917,749 rows
INFO:precursor.bronze:[PASS] alpaca_clean: no null close prices — 0 nulls
INFO:precursor.bronze:[PASS] alpaca_clean: all dates >= 2020-01-01 — 0 pre-2020 rows
INFO:precursor.bronze:[PASS] alpaca_clean: distinct tickers >= 500 — 613 tickers
INFO:precursor.bronze:[PASS] alpaca_clean: ohlc_valid > 99% — 100.00%
INFO:precursor.bronze:[PASS] alpaca_clean: extreme_move < 1% — 0.01%
INFO:precursor.bronze:[PASS] fred_clean: exactly 6 series_ids — 6 found
INFO:precursor.bronze:[PASS] fred_clean: no null values — 0 nulls
INFO:precursor.bronze:[PASS] fred_clean: all range_valid = TRUE — 0 out-of-range rows
INFO:precursor.bronze:[PASS] fred_clean: latest date within 5 days — latest = 2026-04-30
INFO:precursor.bronze:[PASS] sec_clean: row count > 100,000 — 225,346 rows
INFO:precursor.bronze:[PASS] sec_clean: no null transaction_dates — 0 nulls
INFO:precursor.bronze:[

  VALIDATION RESULTS
  [PASS] alpaca_clean: row count > 800,000
         917,749 rows
  [PASS] alpaca_clean: no null close prices
         0 nulls
  [PASS] alpaca_clean: all dates >= 2020-01-01
         0 pre-2020 rows
  [PASS] alpaca_clean: distinct tickers >= 500
         613 tickers
  [PASS] alpaca_clean: ohlc_valid > 99%
         100.00%
  [PASS] alpaca_clean: extreme_move < 1%
         0.01%
  [PASS] fred_clean: exactly 6 series_ids
         6 found
  [PASS] fred_clean: no null values
         0 nulls
  [PASS] fred_clean: all range_valid = TRUE
         0 out-of-range rows
  [PASS] fred_clean: latest date within 5 days
         latest = 2026-04-30
  [PASS] sec_clean: row count > 100,000
         225,346 rows
  [PASS] sec_clean: no null transaction_dates
         0 nulls
  [PASS] sec_clean: date_order_valid > 99%
         100.00%
  [PASS] sec_clean: distinct tickers >= 400
         559 tickers
------------------------------------------------------------
  14 passed  /  0 failed
